In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv('group1_interpolated_complete.csv')
config = pd.read_csv('model_reports/best-cnn-bilstm-rmse.csv')


df['Month'] = pd.to_datetime(df['Month'])
station_i_data = df[(df['Stations'] == 'Stn. I (Central West Bay)') & (df['Month'].dt.year == 2022)]

station_i_data = station_i_data.sort_values('Month')

parameter_cols = [
    'Ammonia (mg/L)',
    'BOD (mg/L)',
    'Dissolved Oxygen (mg/L)',
    'Fecal Coliform, MPN/100ml (Geomean)',
    'Inorganic Phospate (mg/L)',
    'Nitrate (mg/L)',
    'pH (units)'
]

cnn_input_array = station_i_data[parameter_cols].values.reshape(1, 12, 7)

In [ ]:
xlsx_path = "LLDAReport2023.xlsx"
excel_file = pd.ExcelFile(xlsx_path)


all_months_data = []

for sheet in excel_file.sheet_names:
    df_month = excel_file.parse(sheet_name=sheet)
    df_month['Month'] = sheet
    all_months_data.append(df_month)

df_2023 = pd.concat(all_months_data, ignore_index=True)
station_i_2023 = df_2023[df_2023['Laguna Lake Stations'] == 'Stn I (Central West Bay)'].copy()
station_i_2023.columns = station_i_2023.columns.str.strip()
param_cols = ['Ammonia', 'BOD', 'DO', 'Fecal', 'pH', 'Nitrate', 'Inorganic Phosphate']
station_i_2023[param_cols] = station_i_2023[param_cols].interpolate()

true_2023_array = station_i_2023[param_cols].values

print("Final shape:", true_2023_array.shape)

In [ ]:
##ARIMA
import re


def clean_filename(text):
    return re.sub(r'[^a-zA-Z0-9_]', '_', text)

filename_map = {
    'Ammonia (mg/L)': 'Ammonia__mg_L',
    'BOD (mg/L)': 'BOD__mg_L',
    'Dissolved Oxygen (mg/L)': 'Dissolved_Oxygen__mg_L',
    'Fecal Coliform, MPN/100ml (Geomean)': 'Fecal_Coliform__MPN_100ml__Geomean',
    'Inorganic Phospate (mg/L)': 'Inorganic_Phospate__mg_L',
    'Nitrate (mg/L)': 'Nitrate__mg_L',
    'pH (units)': 'pH__units'
}

parameter_cols = list(filename_map.keys())


def load_arima_forecasts_for_station(station_name, config_path="best_model.csv", model_dir="saved_models"):
    station_id = clean_filename(station_name)

    config = pd.read_csv(config_path)
    station_config = config[config["Station"] == station_name]

    forecasts = []

    for param in parameter_cols:
        row = station_config[station_config["Parameter"] == param].iloc[0]
        p, d, q = row['Best_p'], row['Best_d'], row['Best_q']
        param_part = filename_map[param]

        filename = f"{station_id}_{param_part}__p{p}_d{d}_q{q}.pkl"
        filepath = os.path.join(model_dir, filename)

        with open(filepath, "rb") as f:
            model = pickle.load(f)

        forecast = model.forecast(steps=12)
        forecasts.append(forecast.values if hasattr(forecast, 'values') else np.array(forecast))

    return np.column_stack(forecasts) 


arima_forecast_array = load_arima_forecasts_for_station("Stn. I (Central West Bay)")

In [ ]:
print(arima_forecast_array)

In [ ]:

model_path = "cnn_bilstm_station_results_20trials/Stn_I_(Central_West_Bay)_best_model.keras"
data_path = "group1_interpolated_complete.csv"
parameter_cols = [
    'Ammonia (mg/L)', 'BOD (mg/L)', 'Dissolved Oxygen (mg/L)',
    'Fecal Coliform, MPN/100ml (Geomean)', 'Inorganic Phospate (mg/L)',
    'Nitrate (mg/L)', 'pH (units)'
]
lookback = 12
forecast_steps = 12
station_name = "Stn. I (Central West Bay)"

model = load_model(model_path)

df = pd.read_csv(data_path)
df["Month"] = pd.to_datetime(df["Month"])
station_df = df[df["Stations"] == station_name].sort_values("Month")

scaler = MinMaxScaler()
scaler.fit(station_df[parameter_cols])

input_data = station_df[station_df["Month"].dt.year == 2022][parameter_cols]
input_scaled = scaler.transform(input_data)
input_seq = input_scaled.reshape(1, lookback, len(parameter_cols))

forecast_scaled = []

for _ in range(forecast_steps):
    pred = model.predict(input_seq, verbose=0)[0]
    forecast_scaled.append(pred)
    
    input_seq = np.expand_dims(np.vstack([input_seq[0][1:], pred]), axis=0)

forecast_scaled = np.array(forecast_scaled)
forecast_real = scaler.inverse_transform(forecast_scaled)

forecast_df = pd.DataFrame(forecast_real, columns=parameter_cols)
forecast_df["Month"] = pd.date_range(start="2023-01-01", periods=forecast_steps, freq="MS")


print(forecast_df)

In [ ]:
metrics = []

for i, col in enumerate(parameter_cols):
    mae = mean_absolute_error(true_2023_array[:, i], forecast_real[:, i])
    rmse = np.sqrt(mean_squared_error(true_2023_array[:, i], forecast_real[:, i]))
    r2 = r2_score(true_2023_array[:, i], forecast_real[:, i])
    metrics.append({
        "Parameter": col,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })


metrics_df = pd.DataFrame(metrics)
overall_mae = mean_absolute_error(true_2023_array, forecast_real)
overall_rmse = np.sqrt(mean_squared_error(true_2023_array, forecast_real))
overall_r2 = r2_score(true_2023_array, forecast_real)

print(metrics_df)
print("\n📌 Overall Metrics:")
print(f"MAE:  {overall_mae:.4f}")
print(f"RMSE: {overall_rmse:.4f}")
print(f"R²:   {overall_r2:.4f}")

In [ ]:


def plot_forecast_comparison(arima_forecast, cnn_bilstm_forecast, true_data, parameter_index, parameter_name):

    months = np.arange(1, 13)
    
    plt.figure(figsize=(10, 5))
    plt.plot(months, true_data[:, parameter_index], marker='o', label='Actual', linewidth=2)
    plt.plot(months, arima_forecast[:, parameter_index], marker='s', label='ARIMA Forecast')
    plt.plot(months, cnn_bilstm_forecast[:, parameter_index], marker='^', label='CNN-BiLSTM Forecast')

    plt.title(f"Forecast vs Actual: {parameter_name} (2023)")
    plt.xlabel("Month")
    plt.ylabel(parameter_name)
    plt.xticks(months)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

for i, param in enumerate(parameter_cols):
    plot_forecast_comparison(
        arima_forecast=arima_forecast_array,
        cnn_bilstm_forecast=forecast_df[parameter_cols].values,
        true_data=true_2023_array,
        parameter_index=i,
        parameter_name=param
    )

In [ ]:

def evaluate_ensemble(arima_forecast, cnn_forecast, true_data, step=0.05):
    best_w = None
    best_rmse = float('inf')
    results = []

    for w in np.arange(0, 1.01, step):
        ensemble = w * arima_forecast + (1 - w) * cnn_forecast

        y_true = true_data.flatten()
        y_pred = ensemble.flatten()

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)

        results.append({
            'weight_arima': w,
            'weight_cnn': 1 - w,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })

        if rmse < best_rmse:
            best_rmse = rmse
            best_w = w

    return best_w, best_rmse, results


best_weight, best_rmse, metrics_table = evaluate_ensemble(
    arima_forecast_array,
    forecast_df[parameter_cols].values,
    true_2023_array,
    step=0.05
)

print(f"✅ Best ARIMA Weight: {best_weight:.2f}")
print(f"📉 RMSE at Best Weight: {best_rmse:.4f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_ensemble_per_parameter(arima_forecast, cnn_forecast, true_data, step=0.05):
    n_params = arima_forecast.shape[1]
    best_weights = []
    metrics = []

    for i in range(n_params):
        best_rmse = float("inf")
        best_w = None

        for w in np.arange(0, 1.01, step):
            ensemble = w * arima_forecast[:, i] + (1 - w) * cnn_forecast[:, i]
            y_true = true_data[:, i]

            mae = mean_absolute_error(y_true, ensemble)
            rmse = np.sqrt(mean_squared_error(y_true, ensemble))
            r2 = r2_score(y_true, ensemble)

            if rmse < best_rmse:
                best_rmse = rmse
                best_mae = mae
                best_r2 = r2
                best_w = w

        best_weights.append(best_w)
        metrics.append({
            "Parameter": parameter_cols[i],
            "Best ARIMA Weight": best_w,
            "Best CNN-BiLSTM Weight": 1 - best_w,
            "MAE": best_mae,
            "RMSE": best_rmse,
            "R2": best_r2
        })

    return best_weights, pd.DataFrame(metrics)


best_weights, ensemble_metrics_df = evaluate_ensemble_per_parameter(
    arima_forecast=arima_forecast_array,
    cnn_forecast=forecast_df[parameter_cols].values,
    true_data=true_2023_array,
    step=0.05
)

print(ensemble_metrics_df)

In [ ]:
import tensorflow as tf
import os


df_cnn = pd.read_csv("model_reports/best-cnn-rmse.csv")
df_lstm = pd.read_csv("model_reports/best-lstm-rmse.csv")
df_bilstm = pd.read_csv("model_reports/best-bilstm-rmse.csv")

cnn_dir = "reproducible_models/cnn_models/"
lstm_dir = "reproducible_models/lstm_models/"
bilstm_dir = "reproducible_models/bilstm_models/"

models = {
    "cnn": {},
    "lstm": {},
    "bilstm": {}
}

def load_models(model_df, directory, model_type):
    for _, row in model_df.iterrows():
        # Format station name
        station_formatted = row['station'].replace(" ", "_").replace(".", "").replace("(", "").replace(")", "")
        station_formatted = station_formatted.replace("__", "_")
        batch_size = int(row['batch_size'])

        filename = f"{station_formatted}_bs{batch_size}.keras"
        filepath = os.path.join(directory, filename)

        try:
            model = tf.keras.models.load_model(filepath)
            models[model_type][row['station']] = model
            print(f"[{model_type.upper()}] Loaded model: {filename}")
        except Exception as e:
            print(f"[{model_type.upper()}] Failed to load {filename}: {e}")

# load_models(df_cnn, cnn_dir, "cnn")
# load_models(df_lstm, lstm_dir, "lstm")
# load_models(df_bilstm, bilstm_dir, "bilstm")



In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model

# === Load datasets ===
df1 = pd.read_csv("group1_interpolated_complete.csv")
df1["Month"] = pd.to_datetime(df1["Month"])

df2 = pd.read_csv("group2_interpolated_complete.csv")
df2["Month"] = pd.to_datetime(df2["Month"])

# === Parameters ===
parameter_cols = [
    'Ammonia (mg/L)', 'BOD (mg/L)', 'Dissolved Oxygen (mg/L)',
    'Fecal Coliform, MPN/100ml (Geomean)', 'Inorganic Phospate (mg/L)',
    'Nitrate (mg/L)', 'pH (units)'
]
lookback = 12
forecast_steps = 12

# === Results container ===
forecasts = {
    "cnn": {},
    "lstm": {},
    "bilstm": {},
    "cnnbilstm": {},
}

# === Model configs and directories ===
model_sources = {
    "cnn": df_cnn,
    "lstm": df_lstm,
    "bilstm": df_bilstm,
}
model_dirs = {
    "cnn": "reproducible_models/cnn_models/",
    "lstm": "reproducible_models/lstm_models/",
    "bilstm": "reproducible_models/bilstm_models/",
}

# # === Load & forecast CNN, LSTM, BiLSTM ===
# for model_type, df_model in model_sources.items():
#     for _, row in df_model.iterrows():
#         station_name = row['station']

#         if station_name not in models[model_type]:
#             print(f"[{model_type.upper()}] Model not loaded: {station_name}")
#             continue

#         model = models[model_type][station_name]

#         # Try group1
#         station_df = df1[df1["Stations"] == station_name].sort_values("Month")
#         input_data = station_df[station_df["Month"].dt.year == 2022][parameter_cols]

#         # Fallback to group2
#         if input_data.shape[0] != 12:
#             station_df = df2[df2["Stations"] == station_name].sort_values("Month")
#             input_data = station_df[station_df["Month"].dt.year == 2022][parameter_cols]

#         if input_data.shape[0] != 12:
#             print(f"[{model_type.upper()}] Skipping {station_name}: not enough 2022 data.")
#             continue

#         scaler = MinMaxScaler()
#         scaler.fit(station_df[parameter_cols])
#         input_scaled = scaler.transform(input_data)
#         input_seq = input_scaled.reshape(1, lookback, len(parameter_cols))

#         forecast_scaled = []
#         for _ in range(forecast_steps):
#             pred = model.predict(input_seq, verbose=0)[0]
#             forecast_scaled.append(pred)
#             input_seq = np.expand_dims(np.vstack([input_seq[0][1:], pred]), axis=0)

#         forecast_scaled = np.array(forecast_scaled)
#         forecast_real = scaler.inverse_transform(forecast_scaled)

#         forecast_df = pd.DataFrame(forecast_real, columns=parameter_cols)
#         forecast_df["Month"] = pd.date_range(start="2023-01-01", periods=forecast_steps, freq="MS")

#         forecasts[model_type][station_name] = forecast_df
#         print(f"[{model_type.upper()}] Forecast completed for: {station_name}")


# === Load & forecast CNN-BiLSTM ===
cnnbilstm_dir = "cnn-bilstm/"

for filename in os.listdir(cnnbilstm_dir):
    if not filename.endswith(".keras"):
        continue

    # Extract station name from filename
    station_part = filename.replace("_best_model.keras", "")
    station_name = station_part.replace("_", " ").replace("  ", " ")
    station_name = station_name.replace("Stn ", "Stn. ")  # Normalize prefix
    
    if station_name == "Stn. XVI (Sta Rosa)":
        station_name = "Stn. XVI (Sta. Rosa)"
    filepath = os.path.join(cnnbilstm_dir, filename)
    try:
        model = load_model(filepath)
    except Exception as e:
        print(f"[CNN-BiLSTM] Failed to load {filename}: {e}")
        continue

    # Try group1
    station_df = df1[df1["Stations"] == station_name].sort_values("Month")
    input_data = station_df[station_df["Month"].dt.year == 2022][parameter_cols]

    # Fallback to group2
    if input_data.shape[0] != 12:
        station_df = df2[df2["Stations"] == station_name].sort_values("Month")
        input_data = station_df[station_df["Month"].dt.year == 2022][parameter_cols]

    if input_data.shape[0] != 12:
        print(input_data.shape[0])
        print(station_name)
        print(f"[CNN-BiLSTM] Skipping {station_name}: not enough 2022 data.")
        continue

    scaler = MinMaxScaler()
    scaler.fit(station_df[parameter_cols])
    input_scaled = scaler.transform(input_data)
    input_seq = input_scaled.reshape(1, lookback, len(parameter_cols))

    forecast_scaled = []
    for _ in range(forecast_steps):
        pred = model.predict(input_seq, verbose=0)[0]
        forecast_scaled.append(pred)
        input_seq = np.expand_dims(np.vstack([input_seq[0][1:], pred]), axis=0)

    forecast_scaled = np.array(forecast_scaled)
    forecast_real = scaler.inverse_transform(forecast_scaled)

    forecast_df = pd.DataFrame(forecast_real, columns=parameter_cols)
    forecast_df["Month"] = pd.date_range(start="2023-01-01", periods=forecast_steps, freq="MS")

    forecasts["cnnbilstm"][station_name] = forecast_df
    print(f"[CNN-BiLSTM] Forecast completed for: {station_name}")

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import re

# === Load config files ===
config_a = pd.read_csv("arima_model_groupA.csv")
config_b = pd.read_csv("arima_model_groupB.csv")

# === Load datasets ===
df1 = pd.read_csv("group1_interpolated_complete.csv")
df2 = pd.read_csv("group2_interpolated_complete.csv")
df1["Month"] = pd.to_datetime(df1["Month"])
df2["Month"] = pd.to_datetime(df2["Month"])

# === Forecast container ===
arima_forecasts = {}

# === Config and directories ===
configs = {"groupA": config_a, "groupB": config_b}
dirs = {"groupA": "arima_models_groupA", "groupB": "arima_models_groupB"}

# === Parameters ===
forecast_steps = 12

def clean_filename(text):
    return re.sub(r'[^a-zA-Z0-9_]', '_', text)
# === Clean name and build filename ===
def format_arima_filename(station, parameter, p, d, q):
    station_clean = clean_filename(station)
    param_clean = clean_filename(parameter)
    return f"{station_clean}_{param_clean}_p{p}_d{d}_q{q}.pkl"

# === Run forecasts ===
for group_key in ["groupA", "groupB"]:
    config_df = configs[group_key]
    model_dir = dirs[group_key]

    for _, row in config_df.iterrows():
        station = row["Station"]
        parameter = row["Parameter"]
        p, d, q = row["Best_p"], row["Best_d"], row["Best_q"]

        filename = format_arima_filename(station, parameter, p, d, q)
        model_path = os.path.join(model_dir, filename)

        if not os.path.exists(model_path):
            print(f"[ARIMA-{group_key}] Missing: {filename}")
            continue

        # === Load model ===
        try:
            with open(model_path, "rb") as f:
                model = pickle.load(f)
        except Exception as e:
            print(f"[ARIMA-{group_key}] Error loading {filename}: {e}")
            continue

        # === Select data from group1 or group2 ===
        station_df = df1[df1["Stations"] == station].sort_values("Month")
        if parameter not in station_df.columns or station_df[station_df["Month"].dt.year == 2022].shape[0] < 12:
            station_df = df2[df2["Stations"] == station].sort_values("Month")

        if parameter not in station_df.columns or station_df[station_df["Month"].dt.year == 2022].shape[0] < 12:
            print(f"[ARIMA-{group_key}] Skipping {station} - {parameter}: insufficient 2022 data.")
            continue

        # === Forecast ===
        try:
            forecast = model.forecast(steps=forecast_steps)
        except Exception as e:
            print(f"[ARIMA-{group_key}] Forecast error for {station} - {parameter}: {e}")
            continue

        # === Store forecast ===
        if station not in arima_forecasts:
            arima_forecasts[station] = {}
        arima_forecasts[station][parameter] = np.array(forecast)

        print(f"[ARIMA-{group_key}] ✅ Forecasted: {station} - {parameter}")

In [ ]:

ensemble_v1 = pd.read_csv("ensemble_results_all_stations.csv")
ensemble_v2 = pd.read_csv("ensemble_results_all_stations2.csv")
ensemble_forecasts = {}


ensemble_df = pd.concat([ensemble_v1, ensemble_v2], ignore_index=True)


for _, row in ensemble_df.iterrows():
    station = row['Station']
    parameter = row['Parameter']
    arima_w = row['Best ARIMA Weight']
    bilstm_w = row['Best CNN-BiLSTM Weight']

    if station not in arima_forecasts or parameter not in arima_forecasts[station]:
        print(f"[Ensemble] Missing ARIMA forecast for {station} - {parameter}")
        continue
    if station not in forecasts['cnnbilstm']:
        print(f"[Ensemble] Missing CNN-BiLSTM model for {station}")
        continue

    cnn_bilstm_df = forecasts['cnnbilstm'][station]
    if parameter not in cnn_bilstm_df.columns:
        print(f"[Ensemble] Parameter not found in CNN-BiLSTM output: {station} - {parameter}")
        continue

    arima_vals = arima_forecasts[station][parameter]
    cnn_vals = cnn_bilstm_df[parameter].values

    ensemble_vals = arima_w * arima_vals + bilstm_w * cnn_vals

    if station not in ensemble_forecasts:
        ensemble_forecasts[station] = {}
    ensemble_forecasts[station][parameter] = ensemble_vals

    print(f"[Ensemble] ✅ Computed: {station} - {parameter}")

In [ ]:
import json
def convert_ndarrays(obj):
    if isinstance(obj, dict):
        return {k: convert_ndarrays(v) for k, v in obj.items()}
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

clean_data = convert_ndarrays(ensemble_forecasts)
with open("water_forecast.json", "w") as f:
    json.dump(clean_data, f, indent=4)

In [ ]:
print(forecasts['cnn'])

In [ ]:
def plot_all_forecasts(
    station,
    parameter,
    actual_df,
    forecast_dicts,
    forecast_labels
):

    plt.figure(figsize=(10, 5))
    month_nums = np.arange(1, 13)
    plotted = False

    if parameter in actual_df.columns and actual_df.shape[0] == 12:
        plt.plot(month_nums, actual_df[parameter].values, marker='o', label='Actual', linewidth=2)
        plotted = True
    else:
        print(f"[WARN] Skipping actual data: {station} - {parameter}")

    for forecast_dict, label in zip(forecast_dicts, forecast_labels):
        if station not in forecast_dict:
            continue

        forecast_entry = forecast_dict[station]

        if isinstance(forecast_entry, dict):
            if parameter not in forecast_entry:
                continue
            values = forecast_entry[parameter]
        else:
            values = forecast_entry

        if isinstance(values, pd.Series):
            values = values.values
        elif isinstance(values, pd.DataFrame):
            values = values[parameter].values

        if isinstance(values, (list, np.ndarray)) and len(values) == 12:
            style_map = {
                "LSTM": {"linestyle": "--", "linewidth": 3, "marker": "o"},
                "CNN": {"linestyle": "-", "linewidth": 2, "marker": "s"},
                "BiLSTM": {"linestyle": "-.", "linewidth": 2, "marker": "^"},
                "ARIMA": {"linestyle": ":", "linewidth": 2, "marker": "x"},
                "CNN-BiLSTM": {"linestyle": "-", "linewidth": 2, "marker": "D"},
                "Ensemble": {"linestyle": "-", "linewidth": 2, "marker": "*"},
            }

            style = style_map.get(label, {"linestyle": "-", "linewidth": 2, "marker": "x"})
            plt.plot(month_nums, values, label=label, **style)
            plotted = True
        else:
            print(f"[WARN] Skipping {label} for {station} - {parameter} (invalid or missing data)")

    if plotted:
        plt.title(f"{station} - {parameter} (2023 Forecast Comparison)")
        plt.xlabel("Month")
        plt.ylabel(parameter)
        plt.xticks(month_nums)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print(f"[SKIP] No data to plot for {station} - {parameter}")

In [ ]:
import matplotlib.pyplot as plt
# 1. Set station and parameter
station = "Stn I (Central West Bay)"
parameter = "Ammonia (mg/L)"

# 2. Rename actuals DataFrame to match forecast column names
station_i_2023_renamed = station_i_2023.rename(columns={
    "Ammonia": "Ammonia (mg/L)",
    "BOD": "BOD (mg/L)",
    "DO": "Dissolved Oxygen (mg/L)",
    "Fecal": "Fecal Coliform, MPN/100ml (Geomean)",
    "pH": "pH (units)",
    "Nitrate": "Nitrate (mg/L)",
    "Inorganic Phosphate": "Inorganic Phospate (mg/L)"
})

# 3. Forecasts and labels
forecast_sources = [
    arima_forecasts,
    forecasts['cnnbilstm'],
    ensemble_forecasts
]
forecast_labels = [
    "ARIMA",
    "CNN-BiLSTM",
    "Ensemble"
]

# 4. Plot
plot_all_forecasts(
    station=station,
    parameter=parameter,
    actual_df=station_i_2023_renamed,
    forecast_dicts=forecast_sources,
    forecast_labels=forecast_labels
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Load and prepare actuals ===
xlsx_path = "LLDAReport2023.xlsx"
excel_file = pd.ExcelFile(xlsx_path)

all_months_data = []
for sheet in excel_file.sheet_names:
    df_month = excel_file.parse(sheet_name=sheet)
    df_month['Month'] = pd.to_datetime(sheet + " 2023", format="%b %Y")
    all_months_data.append(df_month)

df_2023 = pd.concat(all_months_data, ignore_index=True)
df_2023.columns = df_2023.columns.str.strip()
df_2023['Laguna Lake Stations'] = df_2023['Laguna Lake Stations'].str.strip()

# === Rename actual data columns to match model parameters ===
df_2023 = df_2023.rename(columns={
    "Ammonia": "Ammonia (mg/L)",
    "BOD": "BOD (mg/L)",
    "DO": "Dissolved Oxygen (mg/L)",
    "Fecal": "Fecal Coliform, MPN/100ml (Geomean)",
    "pH": "pH (units)",
    "Nitrate": "Nitrate (mg/L)",
    "Inorganic Phosphate": "Inorganic Phospate (mg/L)"
})

# === Parameters ===
parameter_cols = [
    "Ammonia (mg/L)", "BOD (mg/L)", "Dissolved Oxygen (mg/L)",
    "Fecal Coliform, MPN/100ml (Geomean)", "Inorganic Phospate (mg/L)",
    "Nitrate (mg/L)", "pH (units)"
]

# === Style per model
style_map = {
    "Actual": {"linestyle": "-", "linewidth": 2, "marker": "o"},
    "ARIMA": {"linestyle": ":", "linewidth": 2, "marker": "v"},
    "CNN-BiLSTM": {"linestyle": "-", "linewidth": 2, "marker": "p"},
    "Ensemble": {"linestyle": "-", "linewidth": 2, "marker": "*"},
}

forecast_sources = [
    arima_forecasts,
    forecasts["cnnbilstm"],
    ensemble_forecasts
]
forecast_labels = ["ARIMA", "CNN-BiLSTM", "Ensemble"]

# === Output folder
output_dir = "forecast_plots"
os.makedirs(output_dir, exist_ok=True)

# === Start plotting ===
for raw_station in df_2023['Laguna Lake Stations'].unique():
    station_df = df_2023[df_2023['Laguna Lake Stations'] == raw_station].copy()
    station_df = station_df.sort_values("Month")
    station_df[parameter_cols] = station_df[parameter_cols].interpolate(method="linear", limit_direction="both")

    # Rename station to match keys in forecasts (dot after Stn)
    station = raw_station.replace("Stn ", "Stn. ")

    if raw_station == "Stn. XVI (Sta Rosa)":
        station = "Stn. XVI (Sta. Rosa)"

    for parameter in parameter_cols:
        if station_df[parameter].isna().sum() > 0 or len(station_df) != 12:
            print(f"[SKIP] Incomplete actuals for {station} - {parameter}")
            continue

        month_nums = np.arange(1, 13)
        plt.figure(figsize=(10, 5))

        # === Plot actuals
        plt.plot(month_nums, station_df[parameter].values, label="Actual", **style_map["Actual"])

        # === Plot forecasts
        for forecast_dict, label in zip(forecast_sources, forecast_labels):
            if station not in forecast_dict:
                continue

            forecast_entry = forecast_dict[station]
            if isinstance(forecast_entry, dict):
                if parameter not in forecast_entry:
                    continue
                y_pred = forecast_entry[parameter]
            else:
                y_pred = forecast_entry[parameter].values

            if isinstance(y_pred, (list, np.ndarray)) and len(y_pred) == 12:
                plt.plot(month_nums, y_pred, label=label, **style_map.get(label, {}))

        # === Formatting and save
        plt.title(f"{station} - {parameter} (2023 Forecast Comparison)")
        plt.xlabel("Month")
        plt.ylabel(parameter)
        plt.xticks(month_nums)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        safe_station = station.replace(" ", "_").replace(".", "").replace("(", "").replace(")", "")
        safe_param = parameter.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
        filename = f"{safe_station}_{safe_param}.png"
        plt.savefig(os.path.join(output_dir, filename))
        plt.close()
        print(f"✅ Saved: {filename}")

In [ ]:
print(forecasts["cnn"])

In [ ]:
print(parameter in forecasts['lstm'][station].columns)
print(forecasts['lstm'][station].columns)
print(forecasts['lstm'][station].shape)

In [ ]:
# Compare arrays directly
cnn_vals = forecasts['cnn']["Stn. I (Central West Bay)"]["Ammonia (mg/L)"].values
lstm_vals = forecasts['lstm']["Stn. I (Central West Bay)"]["Ammonia (mg/L)"].values

print("CNN == LSTM?", np.allclose(cnn_vals, lstm_vals, atol=1e-6))

In [ ]:
import pandas as pd

# === Load all cleaned files ===
arima_df = pd.read_csv("station_average_metrics_arima.csv")
cnn_df = pd.read_csv("model_reports/best-cnn-rmse.csv")
lstm_df = pd.read_csv("model_reports/best-lstm-rmse.csv")
bilstm_df = pd.read_csv("model_reports/best-bilstm-rmse.csv")
ensemble_df1 = pd.read_csv("ensemble_results_all_stations.csv")
ensemble_df2 = pd.read_csv("ensemble_results_all_stations2.csv")

# === Clean and standardize columns ===
arima_df_clean = arima_df.rename(columns={"Station": "Station", "MAE": "avg_mae", "RMSE": "avg_rmse"})
arima_df_clean["model"] = "ARIMA"

cnn_df["model"] = "CNN"
lstm_df["model"] = "LSTM"
bilstm_df["model"] = "BiLSTM"

cnn_df_clean = cnn_df.rename(columns={"station": "Station"})[["Station", "avg_mae", "avg_rmse", "model"]]
lstm_df_clean = lstm_df.rename(columns={"station": "Station"})[["Station", "avg_mae", "avg_rmse", "model"]]
bilstm_df_clean = bilstm_df.rename(columns={"station": "Station"})[["Station", "avg_mae", "avg_rmse", "model"]]
ensemble_df = pd.concat([ensemble_df1, ensemble_df2], ignore_index=True)

# === Process Ensemble (average MAE and RMSE per station)
ensemble_avg = ensemble_df.groupby("Station").agg(
    avg_mae=pd.NamedAgg(column="MAE", aggfunc="mean"),
    avg_rmse=pd.NamedAgg(column="RMSE", aggfunc="mean")
).reset_index()
ensemble_avg["model"] = "Ensemble"

# === Load and process CNN-BiLSTM from individual files ===
import os

cnnbilstm_records = []
metrics_dir = "cnn_bilstm_station_results_final/performance_metrics/"  # Change if metrics are in a subfolder
for fname in os.listdir(metrics_dir):
    if fname.endswith("_evaluation_metrics.csv") and "Stn_" in fname:
        df = pd.read_csv(os.path.join(metrics_dir, fname))
        station_name = fname.replace("_evaluation_metrics.csv", "").replace("_", " ")
        avg_mae = df["MAE"].mean()
        avg_rmse = df["RMSE"].mean()
        cnnbilstm_records.append({
            "Station": station_name,
            "avg_mae": avg_mae,
            "avg_rmse": avg_rmse,
            "model": "CNN-BiLSTM"
        })

cnnbilstm_df = pd.DataFrame(cnnbilstm_records)

# === Combine everything ===
combined_df = pd.concat([
    arima_df_clean,
    # cnn_df_clean,
    # lstm_df_clean,
    # bilstm_df_clean,
    cnnbilstm_df,
    ensemble_avg
], ignore_index=True)

# === Save to CSV ===
combined_df.to_csv("station_model_summary_metrics2.csv", index=False)
print("✅ Summary CSV saved as 'station_model_summary_metrics2.csv'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Load Data ===
df = pd.read_csv("station_model_summary_metrics2.csv")

# === Group by model and average the metrics ===
avg_metrics = df.groupby("model")[["avg_mae", "avg_rmse"]].mean().reset_index()

# === Melt for plotting
melted_avg = avg_metrics.melt(
    id_vars="model",
    value_vars=["avg_mae", "avg_rmse"],
    var_name="Metric",
    value_name="Error Value"
)

# === Plot setup
plt.figure(figsize=(10, 6))
sns.set(style="whitegrid")
barplot = sns.barplot(data=melted_avg, x="model", y="Error Value", hue="Metric")

# === Add value labels
for container in barplot.containers:
    barplot.bar_label(container, fmt='%.2f', padding=3)

# === Labels and title
plt.title("Average MAE & RMSE Across All Stations by Model")
plt.ylabel("Error Value")
plt.xlabel("Model")
plt.xticks(rotation=45)
plt.tight_layout()

# === Save or show
plt.savefig("avg_model_error_summary.png")  # Optional
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# === Load Data ===
df = pd.read_csv("station_model_summary_metrics2.csv")

# === Output directory ===
output_dir = "station_model_barplots"
os.makedirs(output_dir, exist_ok=True)

# === Seaborn style ===
sns.set(style="whitegrid")

# === Iterate through stations ===
for station_name in df["Station"].unique():
    df_station = df[df["Station"] == station_name].copy()

    # Melt the DataFrame for grouped barplot
    df_melted = df_station.melt(
        id_vars=["model"],
        value_vars=["avg_mae", "avg_rmse"],
        var_name="Metric",
        value_name="Value"
    )

    # Plot
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(data=df_melted, x="model", y="Value", hue="Metric")

    # Add labels
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3)

    # Titles and formatting
    plt.title(f"{station_name} - Model Performance (MAE & RMSE)")
    plt.ylabel("Error Value")
    plt.xlabel("Model")
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Save plot
    safe_station = (
        station_name.replace(" ", "_")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
    )
    filename = f"{safe_station}_mae_rmse.png"
    plt.savefig(os.path.join(output_dir, filename))


print(f"✅ All plots saved to: {output_dir}")